In [12]:
# installing required libraries
!pip install pennylane kagglehub torch torchvision

In [25]:
# importing required libraries
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.models as models
import pennylane as qml
import kagglehub

In [26]:
# device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [27]:
# downloading the dataset
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
dataset_path = path + "/chest_xray"
print(dataset_path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
/kaggle/input/chest-xray-pneumonia/chest_xray


In [28]:
# image preprocessing
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [29]:
# Loading dataset
train_dataset = torchvision.datasets.ImageFolder(
    dataset_path + "/train",
    transform=transform
)
test_dataset = torchvision.datasets.ImageFolder(
    dataset_path + "/test",
    transform=transform
)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

test_loader = DataLoader(test_dataset,batch_size=32)

In [30]:
# handling class imbalance
class_counts = [1341, 3875]
weights = torch.tensor([
    1/class_counts[0],
    1/class_counts[1]
], dtype=torch.float32)
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [31]:
# Pretrained CNN Feature Extractor
resnet = models.resnet18(pretrained=True)
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features,4)

In [32]:
# Quantum circuit
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [33]:
# Quantum Layer
weight_shapes = {"weights": (2, n_qubits)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

In [34]:
# Hybrid Quantum Model
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = resnet
        self.quantum = qlayer
        self.fc = nn.Linear(4,2)
    def forward(self,x):
        x = self.cnn(x)
        x = self.quantum(x)
        x = self.fc(x)
        return x

In [35]:
# Initializing Model
model = HybridModel()
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [36]:
# training the model
epochs = 5
for epoch in range(epochs):
    total_loss = 0
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print("Epoch:",epoch+1,"Loss:",total_loss)

Epoch: 1 Loss: 88.44928058981895
Epoch: 2 Loss: 64.58155056834221
Epoch: 3 Loss: 49.368465065956116
Epoch: 4 Loss: 40.886637933552265
Epoch: 5 Loss: 34.50923616439104


In [37]:
# testing the accuracy
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted==labels).sum().item()
accuracy = 100 * correct / total
print("Quantum Hybrid Accuracy:",accuracy)

Quantum Hybrid Accuracy: 82.8525641025641
